# 02 - Replay recorded readout traces through the loopback

This is how every hardware result in the paper gets its input: recorded qubit readout traces from the public dataset of
Di Guglielmo et al. (Zenodo record 14427490) are played out of the DAC, through the coax loopback, and recorded by the ADC
exactly as a live readout would be (paper Sec. IV).

Steps:
1. Build a **shuffled, class-balanced** test subset. The dataset file is ordered in blocks of 5,000 shots per class; firing
   it in file order would make any drift during the run look like a class difference.
2. **Predistort** each trace with the board's $(k, c)$ from notebook 01, interpolate it to the DAC envelope rate, and
   scale it to the DAC range.
3. **Replay** and save what comes back.
4. **Check** that the replay preserves the class structure: class-mean trajectories and a matched-filter fidelity, before
   and after the channel.

**You need:** the board as in notebook 01, its `k`/`c` from notebook 01, and the dataset's test split at the readout rate
(`0528_X_test_0_770.npy`, `0528_y_test_0_770.npy`; regenerate from the Zenodo record, see `DATA_MANIFEST.csv`).

In [ ]:
from pathlib import Path

BOARD_HOST = "10.1.9.26"
NS_PORT, PROXY_NAME = 8888, "myqick"

DATA_DIR = Path("..")                       # where the test split is
TEST_X, TEST_Y = "0528_X_test_0_770.npy", "0528_y_test_0_770.npy"
CAL_DIR = Path("outputs/calibration")       # notebook 01's output
OUT_DIR = Path("outputs/replay")

N_SUBSET = 20000      # size of the balanced subset (the paper's headline uses all 100,000 test traces)
SEED = 20260902       # seed of our subsets; the same seed gives the same rows
N_FIRE = 200          # how many of them to replay now (about 2 shots/s)

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import gc
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import Pyro4
from qick import QickConfig
from qick.asm_v2 import AveragerProgramV2
from scipy.interpolate import interp1d

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

## 1. A shuffled, class-balanced subset

Exactly half of each class, drawn without replacement and shuffled with a recorded seed. The source row indices are
saved, so every replayed shot traces back to a row of the dataset.

In [ ]:
def balanced_subset(X, y, n, seed):
    rng = np.random.default_rng(seed)
    idx = np.concatenate([rng.choice(np.where(y == s)[0], size=n // 2, replace=False) for s in (0, 1)])
    rng.shuffle(idx)
    return idx


X_all = np.load(DATA_DIR / TEST_X, mmap_mode="r")
y_all = np.load(DATA_DIR / TEST_Y).astype(int)
idx = balanced_subset(X_all, y_all, N_SUBSET, SEED)
X, y = np.asarray(X_all[idx]).astype(np.float64), y_all[idx]

pos = np.arange(len(y))
print(f"subset {X.shape}, balance {np.bincount(y)}, label runs {1 + np.sum(np.diff(y) != 0)} of {len(y)}, "
      f"order-label correlation {np.corrcoef(pos, y)[0, 1]:+.4f} (want ~0)")
np.save(OUT_DIR / "subset_indices.npy", idx)

## 2. Predistort

$z_\text{sent} = (z - c)/k$, linear interpolation from 770 samples up to an 11,000-sample envelope, then one common scale
factor so the largest value is 99% of the DAC's ±32,766. One factor for the whole set keeps the shots comparable.

In [ ]:
k = complex(np.load(CAL_DIR / f"k_{BOARD_HOST}.npy"))
c = complex(np.load(CAL_DIR / f"c_{BOARD_HOST}.npy"))
print(f"k = {k:.5f} (|k| {abs(k):.5f}, arg {np.degrees(np.angle(k)):.2f} deg), c = {c:.3f}")


def predistort(z_rec, k, c, dac_len=11000, fill=0.99, limit=32766):
    z = (z_rec - c) / k
    xo, xn = np.linspace(0, 1, z.shape[1]), np.linspace(0, 1, dac_len)
    I = interp1d(xo, z.real, axis=1, kind="linear", fill_value="extrapolate")(xn)
    Q = interp1d(xo, z.imag, axis=1, kind="linear", fill_value="extrapolate")(xn)
    env = np.empty((z.shape[0], 2 * dac_len)); env[:, 0::2] = I; env[:, 1::2] = Q
    scale = limit * fill / np.max(np.abs(env))
    return env * scale, scale


def to_complex(x):
    return x[:, 0::2] + 1j * x[:, 1::2]


env, scale = predistort(to_complex(X), k, c)
print(f"envelopes {env.shape}, DAC fill factor {scale:.6f}")

## 3. Replay

Same program as notebook 01 (generator 0 at a 50 MHz carrier, readout 0's decimated stream), triggering only the readout.

In [ ]:
soc = Pyro4.Proxy(Pyro4.locateNS(host=BOARD_HOST, port=NS_PORT).lookup(PROXY_NAME))
soccfg = QickConfig(soc.get_cfg())

BASE_CFG = {"gen_ch": 0, "ro_ch": 0, "nqz": 1, "reps": 1, "final_delay": 100, "readout_length": 1.393,
            "adc_trig_offset": 0.31, "freq": 50, "mode": "oneshot", "stdysel": "zero", "outsel": "product",
            "gain": 1, "phase": 0.0}


class LoopbackProgram(AveragerProgramV2):
    def __init__(self, soccfg, cfg):
        super().__init__(soccfg, reps=cfg["reps"], final_delay=cfg["final_delay"], cfg=cfg)

    def _initialize(self, cfg):
        self.declare_gen(ch=cfg["gen_ch"], nqz=cfg["nqz"])
        self.declare_readout(ch=cfg["ro_ch"], length=cfg["readout_length"])
        self.add_envelope(ch=cfg["gen_ch"], name="env", idata=cfg["waveform_I"], qdata=cfg["waveform_Q"])
        self.add_pulse(ch=cfg["gen_ch"], name="trace", style="arb", envelope="env", gain=cfg["gain"],
                       freq=cfg["freq"], phase=cfg["phase"], outsel=cfg["outsel"], mode=cfg["mode"],
                       stdysel=cfg["stdysel"], ro_ch=cfg["ro_ch"])
        self.add_readoutconfig(ch=cfg["ro_ch"], name="ro", freq=cfg["freq"], gen_ch=cfg["gen_ch"])

    def _body(self, cfg):
        self.pulse(ch=cfg["gen_ch"], name="trace")
        self.send_readoutconfig(ch=cfg["ro_ch"], name="ro", t=0)
        self.trigger(ros=[cfg["ro_ch"]], pins=[0], t=cfg["adc_trig_offset"], width=10)


def pad32(w):
    w = np.asarray(w, dtype=float).reshape(-1)
    n = int(np.ceil(w.size / 32) * 32)
    return np.pad(w, (0, n - w.size))[:n]


rows, t0 = [], time.time()
for i in range(N_FIRE):
    w = pad32(env[i])
    iq = LoopbackProgram(soccfg, dict(BASE_CFG, waveform_I=w[0::2], waveform_Q=w[1::2])).acquire_decimated(soc, rounds=1, progress=False)
    rows.append(np.concatenate([np.asarray(a).reshape(-1) for a in iq]) if isinstance(iq, list) else np.asarray(iq).reshape(-1))
    if i % 50 == 0 or i == N_FIRE - 1:
        el = time.time() - t0
        print(f"  {i + 1}/{N_FIRE}  {(i + 1) / el:.2f} shots/s")
    if i % 200 == 0:
        gc.collect()
acq = np.vstack(rows)
np.save(OUT_DIR / "replayed.npy", acq); np.save(OUT_DIR / "replayed_y.npy", y[:N_FIRE])
json.dump(dict(board=BOARD_HOST, seed=SEED, n_subset=N_SUBSET, n_fired=N_FIRE, k=[k.real, k.imag], c=[c.real, c.imag],
               dac_fill_factor=float(scale)), open(OUT_DIR / "replayed_manifest.json", "w"), indent=1)
print("saved", acq.shape)

## 4. Check the replay

**Class means.** The replayed class means should have the recording's shape, up to the channel's scale and phase.

**Matched filter.** For two states with known mean trajectories $m_0, m_1$ and white noise, the best linear discriminant
projects a trace onto $w = m_1 - m_0$ and thresholds the result. Fidelity here is $1 - \tfrac12[P(1|0) + P(0|1)]$, the
paper's definition. The filter is fit on even shots and scored on odd ones, on the recording and on the replay.

In [ ]:
def matched_filter_fidelity(z, y):
    tr, te = np.arange(len(y)) % 2 == 0, np.arange(len(y)) % 2 == 1
    m0, m1 = z[tr & (y == 0)].mean(axis=0), z[tr & (y == 1)].mean(axis=0)
    w = m1 - m0
    s = np.real(z @ np.conj(w))
    thr = 0.5 * (np.real(m0 @ np.conj(w)) + np.real(m1 @ np.conj(w)))
    pred = (s > thr).astype(int)
    p10 = np.mean(pred[te & (y == 0)] == 1); p01 = np.mean(pred[te & (y == 1)] == 0)
    return 1 - 0.5 * (p10 + p01), p10, p01


z_rec, z_rep, yy = to_complex(X[:N_FIRE]), to_complex(acq)[:, :770], y[:N_FIRE]
for name, z in (("recording", z_rec), ("replay", z_rep)):
    F, p10, p01 = matched_filter_fidelity(z, yy)
    print(f"{name:9s}: matched-filter fidelity {100 * F:.2f}%   P(1|0) {100 * p10:.2f}%   P(0|1) {100 * p01:.2f}%   (n = {len(yy)})")

t = np.arange(770) * 1.808e-3
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharex=True)
for s, col in ((0, "C0"), (1, "C3")):
    ax[0].plot(t, np.abs(z_rec[yy == s].mean(axis=0)), col, label=f"|{s}⟩")
    ax[1].plot(t, np.abs(z_rep[yy == s].mean(axis=0)), col, label=f"|{s}⟩")
ax[0].set_title("recording: |class mean|"); ax[1].set_title("replay: |class mean|")
for a in ax: a.set_xlabel("time (us)")
ax[0].legend(); plt.tight_layout(); plt.show()

**Notes**
- With a few hundred shots the fidelities are rough. The paper's numbers use the full 100,000-trace test split and the
  trained networks, not this matched filter.
- The replay adds noise and the channel's own distortion; compare the two lines above to see what it costs on your
  board (paper Sec. IV and Fig. 4 characterize it on ours).
- `tools/run_loopback.py` and `tools/make_test_subsets.py` in the repository are the script versions of steps 1–3.